# 🎨 AI Comic Book Page Generator

Google Colab notebook — **No API key required**. Uses a free/open-source Stable Diffusion model to generate a 4-panel comic page.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors sentencepiece pillow

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageDraw, ImageFont
import os, textwrap

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = pipe.to(device)
if device == "cuda":
    pipe.enable_attention_slicing()
print("Model loaded successfully!")

In [ ]:
# ✏️ EDIT YOUR STORY HERE

story = """
A college student discovers a mysterious glowing book inside an old college library.
The book suddenly opens by itself.
A magical light comes out of the book.
The student is transported into a beautiful fantasy world.
"""

character = """
A young Indian college student, 20 years old,
short black hair, blue shirt, black trousers,
friendly face, consistent appearance
"""

In [ ]:
scenes = [
f"""{character}
Scene 1: The student is walking inside an old college library.
He notices a mysterious ancient book glowing on a wooden table.
Cinematic comic book illustration, dramatic lighting, detailed background, dynamic composition.""",

f"""{character}
Scene 2: The student carefully opens the mysterious glowing book.
Bright magical light is coming from the pages. He looks surprised.
Cinematic comic book illustration, dramatic lighting, detailed artwork.""",

f"""{character}
Scene 3: A powerful magical light surrounds the student.
Pages of the book are flying through the air. The student looks shocked and amazed.
Fantasy comic book illustration, dynamic action, dramatic lighting.""",

f"""{character}
Scene 4: The student arrives in a spectacular magical fantasy world.
Floating islands, glowing trees and a beautiful sky surround him. He looks amazed.
Epic comic book illustration, cinematic composition, highly detailed."""
]
print("4 comic scenes prepared!")

In [ ]:
os.makedirs("comic_panels", exist_ok=True)
panel_images = []

for i, prompt in enumerate(scenes):
    print(f"Generating Panel {i+1}...")
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted face, extra fingers, deformed hands, duplicate person, bad anatomy, text, watermark, logo",
        num_inference_steps=25,
        guidance_scale=7.5,
        height=512,
        width=512
    ).images[0]
    image.save(f"comic_panels/panel_{i+1}.png")
    panel_images.append(image)

print("All panels generated!")

In [ ]:
PAGE_WIDTH, PAGE_HEIGHT = 1080, 1350
comic_page = Image.new("RGB", (PAGE_WIDTH, PAGE_HEIGHT), "white")
draw = ImageDraw.Draw(comic_page)

try:
    title_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 48)
    panel_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
except:
    title_font = panel_font = ImageFont.load_default()

title = "THE MYSTERIOUS BOOK"
bbox = draw.textbbox((0,0), title, font=title_font)
draw.text(((PAGE_WIDTH-(bbox[2]-bbox[0]))//2, 25), title, fill="black", font=title_font)

positions = [(40,100), (550,100), (40,700), (550,700)]
panel_size = (490,550)

for i, image in enumerate(panel_images):
    image = image.resize(panel_size)
    x, y = positions[i]
    draw.rectangle([x-5,y-5,x+panel_size[0]+5,y+panel_size[1]+5], outline="black", width=8)
    comic_page.paste(image, (x,y))
    draw.text((x+10,y+10), str(i+1), fill="white", stroke_width=3, stroke_fill="black", font=panel_font)

comic_page.save("AI_Comic_Book_Page.png", quality=95)
display(comic_page)

In [ ]:
try:
    bubble_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 22)
except:
    bubble_font = ImageFont.load_default()

def speech_bubble(draw, xy, text, width=210):
    x, y = xy
    lines = textwrap.wrap(text, width=20)
    line_height = 27
    height = len(lines) * line_height + 25
    draw.rounded_rectangle([x,y,x+width,y+height], radius=20, fill="white", outline="black", width=3)
    draw.polygon([(x+30,y+height),(x+10,y+height+30),(x+65,y+height)], fill="white", outline="black")
    for i, line in enumerate(lines):
        draw.text((x+12,y+10+i*line_height), line, fill="black", font=bubble_font)

draw = ImageDraw.Draw(comic_page)
speech_bubble(draw, (70,130), "What is this strange book?")
speech_bubble(draw, (580,130), "It is glowing!")
speech_bubble(draw, (70,730), "What's happening to me?")
speech_bubble(draw, (580,730), "Where am I?!")

comic_page.save("Final_Comic_Book.png", quality=95)
display(comic_page)

In [ ]:
from google.colab import files
files.download("Final_Comic_Book.png")